# LRTIA - Long-Range Token Influence Analyzer

**Run on Colab Pro with GPU (T4 or A100)**

Runtime > Change runtime type > GPU

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes scipy

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from typing import List, Dict
import random
import re
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Choose Your Model

**Recommended for real long-range effects:**
- `mistralai/Mistral-7B-v0.1` - Best quality, fits on T4 with 4-bit
- `meta-llama/Llama-2-7b-hf` - Needs HF token, great quality
- `EleutherAI/pythia-2.8b` - No auth needed, decent quality
- `microsoft/phi-2` - Small but capable (2.7B)

In [ ]:
# === CHOOSE MODEL ===
MODEL_NAME = "mistralai/Mistral-7B-v0.1"  # Best option - real LLM!
# MODEL_NAME = "EleutherAI/pythia-2.8b"  # Alternative - no auth needed
# MODEL_NAME = "microsoft/phi-2"  # Smaller but capable

# For Llama-2, uncomment and add your token:
# MODEL_NAME = "meta-llama/Llama-2-7b-hf"
# from huggingface_hub import login
# login(token="your_hf_token_here")

USE_4BIT = True  # Enable for 7B models on T4 (16GB)

print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT and device == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map="auto" if device == "cuda" else None,
    )

model.eval()
print(f"Loaded! Parameters: {model.num_parameters():,}")

## 2. Test Corpus: Intact vs Shuffled Natural Text

In [ ]:
# Longer passages for better long-range testing
PASSAGES = [
    """The Amazon rainforest is the world's largest tropical rainforest, covering approximately 5.5 million square kilometers. It spans across nine countries in South America, with the majority located in Brazil. The forest is often called the lungs of the Earth because it produces about 20 percent of the world's oxygen. Scientists estimate that the Amazon is home to approximately 390 billion individual trees, representing around 16,000 different species. The biodiversity found here is unmatched anywhere else on the planet. The Amazon River, which flows through the heart of the forest, is the second longest river in the world. It carries more water than any other river system, accounting for roughly 20 percent of all freshwater that flows into the world's oceans. The river and its tributaries support an incredible variety of aquatic life, including pink river dolphins, giant otters, and piranhas. More than 3,000 species of fish have been identified in its waters, with scientists believing many more remain undiscovered. Indigenous peoples have inhabited the Amazon for at least 11,000 years. Today, approximately 400 distinct indigenous groups live within the forest, speaking more than 300 different languages. These communities have developed sophisticated knowledge of the forest's medicinal plants and sustainable harvesting techniques. Their traditional practices have helped maintain the forest's ecological balance for generations. Deforestation poses the greatest threat to the Amazon's survival. Each year, thousands of square kilometers of forest are cleared for cattle ranching, soybean farming, and logging. This destruction releases massive amounts of carbon dioxide into the atmosphere, contributing to global climate change. Scientists warn that continued deforestation could push the Amazon past a tipping point, transforming large portions of the rainforest into savanna. Conservation efforts have intensified in recent decades. Brazil has established numerous protected areas and indigenous territories that limit development. International organizations have invested billions of dollars in preservation programs. Satellite monitoring now tracks deforestation in real time, enabling faster responses to illegal clearing.""",
    
    """Marie Curie was born Maria Sklodowska in Warsaw, Poland, on November 7, 1867. She grew up in a family that valued education, despite living under Russian occupation that restricted Polish culture and language. Her father was a physics and mathematics teacher, and her mother ran a prestigious boarding school. From an early age, Marie showed exceptional intelligence and a passion for learning. At the time, women in Poland were not permitted to attend university. Marie worked as a governess for several years to help fund her older sister's medical studies in Paris. In exchange, her sister later helped support Marie's own education. In 1891, at age 24, Marie finally moved to Paris to study physics and mathematics at the Sorbonne, one of the few European universities that admitted women. In Paris, Marie lived in a tiny attic apartment and often survived on little more than bread and chocolate. Despite these hardships, she excelled in her studies, earning degrees in both physics and mathematics. It was during this time that she met Pierre Curie, a professor at the School of Physics. They shared a passion for scientific research and were married in 1895. Their partnership would prove to be one of the most productive collaborations in scientific history. Marie became fascinated by Henri Becquerel's recent discovery of mysterious rays emitted by uranium. She decided to investigate these rays for her doctoral thesis. Working in a converted shed with minimal equipment, she and Pierre made breakthrough after breakthrough. Marie coined the term radioactivity to describe the phenomenon. In 1898, she announced the discovery of two new elements: polonium, named after her homeland, and radium. In 1903, Marie Curie became the first woman to win a Nobel Prize, sharing the physics prize with Pierre and Henri Becquerel. Tragically, Pierre was killed in a street accident in 1906. Despite her grief, Marie continued their research and took over his teaching position at the Sorbonne. She became the university's first female professor. In 1911, she won a second Nobel Prize, this time in chemistry. She remains the only person to win Nobel Prizes in two different sciences.""",

    """The Great Wall of China stands as one of humanity's most impressive architectural achievements, stretching across thousands of miles of mountains, deserts, and grasslands. Construction of defensive walls in China began as early as the 7th century BC, when individual states built barriers to protect their territories from neighboring rivals and nomadic tribes from the north. These early walls were typically made of packed earth and were relatively modest in scale. The first emperor of unified China, Qin Shi Huang, initiated the most ambitious wall-building project in 221 BC. After conquering the warring states and establishing the Qin Dynasty, he ordered the connection and extension of existing walls to create a continuous defensive barrier. Hundreds of thousands of soldiers, peasants, and prisoners were conscripted for this massive undertaking. Working conditions were brutal, and countless laborers died during construction. Their bodies were sometimes buried within the wall itself. Subsequent dynasties continued to maintain, rebuild, and extend the wall over the following centuries. The Han Dynasty pushed the wall westward to protect the Silk Road trade routes. The wall fell into disrepair during some periods and was extensively renovated during others. The most iconic sections that tourists visit today were built during the Ming Dynasty, from the 14th to 17th centuries. These sections feature brick and stone construction, watchtowers, and garrison stations. The Great Wall is not a single continuous structure but rather a series of walls and fortifications. Its total length, including all branches and secondary sections, extends approximately 13,000 miles. The main wall follows the historical northern borders of China, traversing diverse terrain from the mountains near Beijing to the deserts of the Gobi. Watchtowers placed at regular intervals allowed soldiers to send smoke signals during the day and fire signals at night to warn of approaching enemies. Today, the Great Wall attracts millions of visitors from around the world each year. Several sections near Beijing have been restored and developed for tourism. However, many remote sections continue to crumble from neglect and erosion.""",

    """Albert Einstein was born in Ulm, Germany, on March 14, 1879. His family moved to Munich when he was an infant, where his father and uncle founded an electrical equipment company. Young Albert showed an early fascination with invisible forces, reportedly becoming captivated by a compass at age five. He wondered what unseen power could make the needle always point north. Einstein struggled with the rigid, authoritarian style of German schools. He excelled in physics and mathematics but chafed at rote memorization and strict discipline. At age 15, he dropped out of school and moved to Switzerland, where he eventually gained admission to the Swiss Federal Polytechnic in Zurich. After graduating, Einstein could not find a teaching position and took a job at the Swiss Patent Office in Bern. This seemingly mundane work proved fortuitous. The job left him time to think about physics, and he later credited the patent office with teaching him to question assumptions and express ideas precisely. In 1905, Einstein published four groundbreaking papers that would revolutionize physics. One explained the photoelectric effect using quantum theory, work that would later earn him the Nobel Prize. Another provided evidence for the existence of atoms by explaining Brownian motion. The third introduced special relativity, fundamentally changing our understanding of space and time. The fourth derived the famous equation E equals mc squared, revealing the equivalence of mass and energy. Einstein spent the next decade developing his general theory of relativity, which reimagined gravity not as a force but as a curvature of spacetime caused by mass and energy. The theory made startling predictions: that light would bend around massive objects, that time would slow in strong gravitational fields, and that the universe itself might be expanding. When a 1919 solar eclipse confirmed his prediction of light bending, Einstein became an international celebrity overnight.""",
]

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]

def create_corpus(passages, n_shuffles=3):
    """Create intact and multiple shuffled versions."""
    docs = []
    for i, passage in enumerate(passages):
        # Original intact version
        docs.append({
            "doc_id": f"intact_{i:03d}",
            "population": "intact",
            "text": " ".join(passage.split())
        })
        # Multiple shuffled versions with different seeds
        sentences = split_sentences(passage)
        for s in range(n_shuffles):
            rng = random.Random(42 + i * 100 + s)
            shuffled = sentences.copy()
            rng.shuffle(shuffled)
            docs.append({
                "doc_id": f"shuffled_{i:03d}_{s}",
                "population": "shuffled",
                "text": " ".join(shuffled)
            })
    return docs

corpus = create_corpus(PASSAGES, n_shuffles=4)
print(f"Created {len(corpus)} documents")
print(f"  - {sum(1 for d in corpus if d['population']=='intact')} intact")
print(f"  - {sum(1 for d in corpus if d['population']=='shuffled')} shuffled")

lengths = [len(tokenizer.encode(d['text'])) for d in corpus]
print(f"\nToken lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)//len(lengths)}")

## 3. Analysis Functions

In [ ]:
@torch.no_grad()
def get_token_logprobs(token_ids: List[int], positions: List[int]) -> np.ndarray:
    """Get log probabilities of actual next tokens at given positions."""
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]  # [seq_len, vocab]
    log_probs = torch.log_softmax(logits, dim=-1)
    
    results = []
    for pos in positions:
        if pos + 1 < len(token_ids):
            next_token = token_ids[pos + 1]
            results.append(log_probs[pos, next_token].cpu().item())
    return np.array(results)


def mask_span(token_ids: List[int], start: int, end: int, mask_id: int) -> List[int]:
    """Replace span with mask tokens."""
    result = token_ids.copy()
    for i in range(start, min(end, len(result))):
        result[i] = mask_id
    return result


def analyze_document(
    text: str,
    distances: List[int],
    span_width: int = 20,
    target_interval: int = 60,
    region_length: int = 30,
    min_context: int = 64,
) -> List[Dict]:
    """Measure effect of masking spans at various distances."""
    token_ids = tokenizer.encode(text)
    n_tokens = len(token_ids)
    mask_id = tokenizer.eos_token_id or tokenizer.pad_token_id or 0
    
    results = []
    
    # Generate target positions
    for target_start in range(min_context, n_tokens - region_length, target_interval):
        target_end = min(target_start + region_length, n_tokens - 1)
        target_positions = list(range(target_start, target_end))
        
        if len(target_positions) < 5:
            continue
        
        # Original predictions
        orig_lp = get_token_logprobs(token_ids, target_positions)
        
        for distance in distances:
            span_center = target_start - distance
            span_start = max(0, span_center - span_width // 2)
            span_end = span_start + span_width
            
            if span_start < 0 or span_end >= target_start - 5:
                continue
            
            # Masked predictions
            masked_ids = mask_span(token_ids, span_start, span_end, mask_id)
            masked_lp = get_token_logprobs(masked_ids, target_positions)
            
            # Delta NLL: positive means masking hurt predictions
            delta_nll = float(np.mean(-masked_lp) - np.mean(-orig_lp))
            
            results.append({
                "distance": distance,
                "delta_nll": delta_nll,
            })
    
    return results

## 4. Run Analysis

In [ ]:
# Configuration - test longer distances for real LLMs
DISTANCES = [32, 64, 128, 256, 384, 512]
SPAN_WIDTH = 20
TARGET_INTERVAL = 50  # More samples
REGION_LENGTH = 25

all_results = []

for doc in tqdm(corpus, desc="Analyzing"):
    try:
        doc_results = analyze_document(
            doc["text"],
            distances=DISTANCES,
            span_width=SPAN_WIDTH,
            target_interval=TARGET_INTERVAL,
            region_length=REGION_LENGTH,
        )
        for r in doc_results:
            r["doc_id"] = doc["doc_id"]
            r["population"] = doc["population"]
            all_results.append(r)
    except Exception as e:
        print(f"Error on {doc['doc_id']}: {e}")

df = pd.DataFrame(all_results)
print(f"\nCollected {len(df)} measurements")
print("\nSamples per condition:")
print(df.groupby(['population', 'distance']).size().unstack(fill_value=0))

## 5. Results

In [ ]:
print("=" * 70)
print(f"RESULTS: {MODEL_NAME}")
print("=" * 70)
print("\nMean delta_nll by population and distance:")
pivot = df.pivot_table(values='delta_nll', index='distance', columns='population', aggfunc='mean')
print(pivot.round(4))

print("\nOverall means:")
means = df.groupby('population')['delta_nll'].mean()
print(means.round(4))

print(f"\nDifference (intact - shuffled): {means.get('intact', 0) - means.get('shuffled', 0):.4f}")

In [ ]:
# Plot memory curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Memory curves
ax = axes[0]
colors = {'intact': '#2ecc71', 'shuffled': '#e74c3c'}

for pop in ['intact', 'shuffled']:
    pop_df = df[df['population'] == pop]
    means = pop_df.groupby('distance')['delta_nll'].mean()
    stds = pop_df.groupby('distance')['delta_nll'].std()
    counts = pop_df.groupby('distance').size()
    sems = stds / np.sqrt(counts)
    
    ax.errorbar(means.index, means.values, yerr=sems.values,
                marker='o', capsize=5, label=pop, 
                color=colors[pop], linewidth=2, markersize=8)

ax.set_xlabel('Distance (tokens)', fontsize=12)
ax.set_ylabel('Delta NLL (effect of masking)', fontsize=12)
ax.set_title(f'Memory Curves: {MODEL_NAME.split("/")[-1]}', fontsize=14)
ax.set_xscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Right: Difference plot
ax = axes[1]
intact_means = df[df['population']=='intact'].groupby('distance')['delta_nll'].mean()
shuffled_means = df[df['population']=='shuffled'].groupby('distance')['delta_nll'].mean()
diff = intact_means - shuffled_means

ax.bar(range(len(diff)), diff.values, color=['#3498db' if d > 0 else '#95a5a6' for d in diff.values])
ax.set_xticks(range(len(diff)))
ax.set_xticklabels(diff.index)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Distance (tokens)', fontsize=12)
ax.set_ylabel('Intact - Shuffled', fontsize=12)
ax.set_title('Difference (positive = intact uses context more)', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('lrtia_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSaved plot to lrtia_results.png")

In [ ]:
# Half-life calculation
print("\n" + "=" * 70)
print("HALF-LIFE ANALYSIS")
print("=" * 70)

from scipy.optimize import curve_fit

def exp_decay(x, a, tau):
    return a * np.exp(-x / tau)

for pop in ['intact', 'shuffled']:
    pop_df = df[df['population'] == pop]
    means = pop_df.groupby('distance')['delta_nll'].mean().sort_index()
    
    x = np.array(means.index)
    y = np.array(means.values)
    
    # Filter positive values for fitting
    mask = y > 0
    if mask.sum() < 3:
        print(f"{pop}: Not enough positive values to fit")
        continue
    
    try:
        popt, _ = curve_fit(exp_decay, x[mask], y[mask], p0=[y[mask].max(), 100], maxfev=5000)
        a, tau = popt
        half_life = tau * np.log(2)
        
        # R-squared
        y_pred = exp_decay(x[mask], *popt)
        ss_res = np.sum((y[mask] - y_pred) ** 2)
        ss_tot = np.sum((y[mask] - np.mean(y[mask])) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        print(f"\n{pop.upper()}:")
        print(f"  Peak effect: {y[0]:.4f}")
        print(f"  Decay constant (tau): {tau:.1f} tokens")
        print(f"  Half-life: {half_life:.1f} tokens")
        print(f"  R-squared: {r2:.3f}")
    except Exception as e:
        print(f"{pop}: Fitting failed - {e}")

In [ ]:
# Save results
df.to_csv('lrtia_results.csv', index=False)
print("Results saved to lrtia_results.csv")

# Download link for Colab
try:
    from google.colab import files
    files.download('lrtia_results.csv')
    files.download('lrtia_results.png')
except:
    pass

## 6. Interpretation

**What to look for:**

1. **Both curves should decay** - effect decreases with distance (expected)

2. **Intact should be HIGHER than shuffled** especially at longer distances
   - This means the model relies on distant context more when text is coherent
   
3. **Intact should have LONGER half-life** than shuffled
   - Coherent text maintains context relevance over longer distances

If you don't see differentiation:
- Try even longer distances (768, 1024)
- Try a larger model
- The model may simply not leverage long-range coherence for this task